# 06 — Key Research Papers: Code Implementations

Implement the core idea from each foundational paper as working code.

**Papers:** ReAct · Reflexion · Generative Agents Memory · Voyager Skills · SWE-agent ACI

**Prerequisites:** `pip install openai python-dotenv rich`

In [ ]:
import os, json, time, math
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI
from rich import print as rprint
from rich.panel import Panel

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('Setup complete')

## Paper 1: ReAct — Thought → Action → Observation Loop

Core idea: Interleave reasoning traces with tool use actions.
Paper: https://arxiv.org/abs/2210.03629

In [ ]:
# Mock tools for ReAct demo
def search(query: str) -> str:
    data = {
        "react paper": "ReAct (2022) by Yao et al. Synergizes reasoning and acting in language models using Thought-Action-Observation traces.",
        "langchain": "LangChain is a framework for building LLM applications. Released in 2022 by Harrison Chase.",
        "default": f"Results for '{query}': [Retrieved relevant information]"
    }
    for k, v in data.items():
        if k.lower() in query.lower():
            return v
    return data["default"]

REACT_SYSTEM = """You are an agent using the ReAct (Reasoning + Acting) approach.
For each step:
1. Think out loud: 'Thought: [your reasoning]'
2. Take an action: use the search tool OR say FINAL ANSWER: [your answer]
Be explicit about your reasoning before every action."""

TOOLS = [{
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search for information on a topic.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"]
        }
    }
}]

def react_agent(question: str, max_steps: int = 5) -> str:
    """Classic ReAct implementation: Thought → Action → Observation loop."""
    print(f"REACT AGENT")
    print(f"Question: {question}\n")
    
    messages = [
        {"role": "system", "content": REACT_SYSTEM},
        {"role": "user",   "content": question}
    ]
    
    for step in range(1, max_steps + 1):
        print(f"Step {step}:")
        response = client.chat.completions.create(
            model="gpt-4o-mini", messages=messages,
            tools=TOOLS, tool_choice="auto"
        )
        msg = response.choices[0].message
        
        if not msg.tool_calls:
            # FINAL ANSWER
            content = msg.content
            if content:
                print(f"  Thought: Agent decided it has enough info")
                print(f"  FINAL: {content}")
            return content
        
        # TOOL USE
        tc = msg.tool_calls[0]
        args = json.loads(tc.function.arguments)
        observation = search(args["query"])
        
        print(f"  Action: search('{args['query']}')")        
        print(f"  Observation: {observation[:80]}")
        
        messages.append(msg)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": observation})
    
    return "Max steps reached."

react_agent("What is the ReAct paper and which framework implemented it first?")

## Paper 2: Reflexion — Verbal Reinforcement Learning

Core idea: After failure, generate a verbal critique → store in memory → use on retry.
Paper: https://arxiv.org/abs/2303.11366

In [ ]:
class ReflexionAgent:
    """Reflexion: learns from failures via verbal self-reflection stored as memory."""
    
    def __init__(self):
        self.episodic_memory = []  # Verbal reflections from past failures
    
    def actor(self, task: str) -> str:
        """Generates a response using past reflections."""
        reflection_context = ""
        if self.episodic_memory:
            reflection_context = "\n\nLearned from past failures:\n" + "\n".join(
                f"Episode {i+1}: {r}" for i, r in enumerate(self.episodic_memory)
            )
        
        return client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Complete the task. Apply lessons from past failures." + reflection_context},
                {"role": "user",   "content": task}
            ]
        ).choices[0].message.content
    
    def evaluator(self, task: str, response: str) -> tuple[bool, str]:
        """Binary evaluation: did the agent succeed? And what was wrong?"""
        result = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content":
                f"Task: {task}\nResponse: {response}\n\n"
                "Did this response fully satisfy the task? Score 0.0-1.0.\n"
                "If not perfect, what is the main failure?\n"
                "Format: SCORE: 0.X\nFAILURE: [what went wrong]"
            }]
        ).choices[0].message.content
        
        score, failure = 0.5, "Unknown failure"
        for line in result.split('\n'):
            if line.startswith('SCORE:'):
                try: score = float(line.split(':')[1].strip())
                except: pass
            if line.startswith('FAILURE:'):
                failure = line.replace('FAILURE:', '').strip()
        
        return score >= 0.85, failure
    
    def self_reflect(self, task: str, response: str, failure: str) -> str:
        """Generate verbal reflection — the KEY insight of Reflexion."""
        return client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content":
                f"Task: {task}\nMy response: {response}\nFailure: {failure}\n\n"
                "In one specific sentence: what exactly should I do differently next time?"
            }]
        ).choices[0].message.content
    
    def run(self, task: str, max_episodes: int = 3):
        """Full Reflexion loop across multiple episodes."""
        print(f"REFLEXION AGENT")
        print(f"Task: {task}")
        
        for episode in range(1, max_episodes + 1):
            print(f"\n--- Episode {episode} ---")
            
            response = self.actor(task)
            print(f"Response: {response[:120]}")
            
            success, failure = self.evaluator(task, response)
            print(f"Success: {success} | Failure note: {failure[:70]}")
            
            if success:
                print(f"✅ Task completed successfully in episode {episode}!")
                return response
            
            reflection = self.self_reflect(task, response, failure)
            self.episodic_memory.append(reflection)
            print(f"Reflection stored: {reflection[:80]}")
        
        print(f"\nCompleted {max_episodes} episodes. Learned {len(self.episodic_memory)} lessons.")

agent = ReflexionAgent()
agent.run(
    task="Write a haiku about artificial intelligence. Must have exactly 5-7-5 syllable structure.",
    max_episodes=3
)

## Paper 3: Generative Agents Memory — The Memory Stream

Core idea: Memory scored by recency + importance + relevance. Retrieved by blended score.
Paper: https://arxiv.org/abs/2304.03442

In [ ]:
@dataclass
class Memory:
    content: str
    timestamp: float = field(default_factory=time.time)
    importance: float = 5.0  # 1-10 rated by LLM

class GenerativeAgentMemory:
    """
    Implements the memory system from Generative Agents paper.
    Retrieval score = α₁×recency + α₂×importance + α₃×relevance
    """
    
    def __init__(self, decay_factor: float = 0.99):
        self.stream: list[Memory] = []  # All memories in chronological order
        self.decay = decay_factor
    
    def rate_importance(self, memory_content: str) -> float:
        """LLM rates importance on 1-10 scale."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content":
                f"Rate the importance of this memory on a scale of 1-10.\n"
                f"1=trivial routine, 10=emotionally significant/life-changing\n"
                f"Memory: {memory_content}\nReturn ONLY a number."
            }]
        ).choices[0].message.content.strip()
        try:
            return min(10.0, max(1.0, float(response)))
        except:
            return 5.0
    
    def add_memory(self, content: str):
        """Add to memory stream, rate importance automatically."""
        importance = self.rate_importance(content)
        memory = Memory(content=content, importance=importance)
        self.stream.append(memory)
        print(f"  Memory stored (importance={importance:.1f}): {content[:60]}")
    
    def recency_score(self, memory: Memory) -> float:
        """Exponential decay based on time since memory was created."""
        hours_ago = (time.time() - memory.timestamp) / 3600
        return self.decay ** hours_ago
    
    def relevance_score(self, memory: Memory, query: str) -> float:
        """Simplified relevance: keyword overlap (in production: cosine similarity of embeddings)."""
        query_words = set(query.lower().split())
        memory_words = set(memory.content.lower().split())
        overlap = len(query_words & memory_words)
        return min(1.0, overlap / max(len(query_words), 1))
    
    def retrieve(self, query: str, top_k: int = 3, 
                 w_recency=0.3, w_importance=0.4, w_relevance=0.3) -> list[Memory]:
        """Retrieve top-k memories by blended score."""
        scored = []
        for mem in self.stream:
            r = self.recency_score(mem)
            i = mem.importance / 10.0
            v = self.relevance_score(mem, query)
            score = w_recency * r + w_importance * i + w_relevance * v
            scored.append((mem, score, r, i, v))
        
        scored.sort(key=lambda x: x[1], reverse=True)
        
        print(f"\nRetrieval for: '{query}'")
        print(f"{'Memory':<50} {'Score':>6} (rec={w_recency}, imp={w_importance}, rel={w_relevance})")
        print("-" * 70)
        for mem, score, r, i, v in scored[:top_k + 2]:
            marker = "✅" if (mem, score, r, i, v) in scored[:top_k] else "  "
            print(f"{marker} {mem.content[:48]:<50} {score:.3f} (r={r:.2f},i={i:.2f},v={v:.2f})")
        
        return [s[0] for s in scored[:top_k]]

# Demo
memory = GenerativeAgentMemory()

print("Adding memories...")
memories_to_add = [
    "Had coffee in the morning",
    "Got a promotion at work today — huge career milestone!",
    "Went to the grocery store",
    "Had an important meeting about the AI project roadmap",
    "Watched a documentary about neural networks",
    "Received an urgent call about a production outage",
]
for m in memories_to_add:
    memory.add_memory(m)

# Retrieve
retrieved = memory.retrieve("work career AI project", top_k=3)
print("\nRetrieved context for reasoning:")
for m in retrieved:
    print(f"  → {m.content}")

## Paper 4: Voyager — Skill Library (Procedural Memory)

Core idea: Store successful action sequences as reusable skills. Retrieve by similarity.
Paper: https://arxiv.org/abs/2305.16291

In [ ]:
@dataclass
class Skill:
    name: str
    description: str
    code: str          # The reusable implementation
    success_count: int = 1

class SkillLibrary:
    """
    Voyager-inspired skill library: agent discovers effective tool sequences,
    stores them, and retrieves/reuses for similar future tasks.
    """
    
    def __init__(self):
        self.skills: list[Skill] = []
    
    def _skill_matches(self, skill: Skill, task: str) -> float:
        """Simplified matching: keyword overlap (production uses embedding cosine sim)."""
        task_words = set(task.lower().split())
        skill_words = set((skill.name + ' ' + skill.description).lower().split())
        overlap = len(task_words & skill_words)
        return overlap / max(len(task_words), 1)
    
    def retrieve_skill(self, task: str) -> Skill | None:
        """Find most relevant skill for current task."""
        if not self.skills:
            return None
        scored = [(s, self._skill_matches(s, task)) for s in self.skills]
        best_skill, best_score = max(scored, key=lambda x: x[1])
        return best_skill if best_score > 0.2 else None
    
    def acquire_skill(self, task: str) -> tuple[str, Skill]:
        """Generate and store a new skill for this type of task."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content":
                f"Task: {task}\n\n"
                "Create a reusable Python function that solves this type of task.\n"
                "The function should be general enough to reuse for similar tasks.\n"
                "Return JSON: {\"name\": \"function_name\", \"description\": \"what it does\", \"code\": \"def function_name(...): ...\"}\n"
                "Return ONLY valid JSON."
            }]
        ).choices[0].message.content
        
        try:
            if '```' in response:
                response = response.split('```')[1].replace('json', '').strip()
            data = json.loads(response)
            skill = Skill(**data)
            self.skills.append(skill)
            return "new_skill", skill
        except:
            skill = Skill(name="generic_solver", description=task, code=f"# solves: {task}")
            self.skills.append(skill)
            return "new_skill", skill
    
    def solve(self, task: str) -> str:
        """Attempt to reuse existing skill or acquire a new one."""
        print(f"\nTask: {task}")
        
        existing = self.retrieve_skill(task)
        if existing:
            existing.success_count += 1
            print(f"✅ Reused skill: '{existing.name}' (used {existing.success_count}x)")
            print(f"   Description: {existing.description}")
            return f"Used skill '{existing.name}'"
        else:
            print("📚 No matching skill found — acquiring new skill...")
            _, new_skill = self.acquire_skill(task)
            print(f"✅ Acquired: '{new_skill.name}'")
            print(f"   Stored in library for future reuse")
            return f"Acquired and used new skill '{new_skill.name}'"

library = SkillLibrary()

# First time: must acquire new skills
library.solve("Calculate compound interest for a savings account")
library.solve("Search the web for Python documentation")
library.solve("Write a Python function to sort a list")

print(f"\nLibrary now has {len(library.skills)} skills")

# Second time: should reuse
print("\n--- New session, similar tasks ---")
library.solve("Compute interest rate for investment account")  # matches compound interest
library.solve("Sort a Python dictionary by values")            # matches sort skill

## Paper 5: SWE-agent — ACI (Agent-Computer Interface) Design

Core idea: Purpose-built, LLM-friendly tools dramatically outperform raw shell access.
Paper: https://arxiv.org/abs/2405.15793

In [ ]:
# Simulate: Raw shell tools vs purpose-built ACI tools
# SWE-agent found 48% improvement from raw bash → custom ACI

# BAD: Raw, generic tools (like giving the agent raw bash)
RAW_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_bash",
            "description": "Run a bash command.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"]
            }
        }
    }
]

# GOOD: Purpose-built ACI tools (like SWE-agent)
ACI_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "open_file",
            "description": "Open a file and view its contents with line numbers. Optionally specify a line range.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "File path relative to project root"},
                    "start_line": {"type": "integer", "description": "Optional: first line to show"},
                    "end_line": {"type": "integer", "description": "Optional: last line to show"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_in_file",
            "description": "Search for a pattern in a specific file. Returns matching lines with numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file": {"type": "string"},
                    "pattern": {"type": "string", "description": "String pattern to search for"}
                },
                "required": ["file", "pattern"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "edit_lines",
            "description": "Replace a specific line range in a file with new content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "file": {"type": "string"},
                    "start_line": {"type": "integer"},
                    "end_line": {"type": "integer"},
                    "new_content": {"type": "string"}
                },
                "required": ["file", "start_line", "end_line", "new_content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run the test suite and return pass/fail status with error messages.",
            "parameters": {
                "type": "object",
                "properties": {
                    "test_file": {"type": "string", "description": "Optional: specific test file to run"}
                }
            }
        }
    }
]

def show_aci_advantage():
    """Ask the LLM to plan how it would use each tool set, showing quality difference."""
    task = "Fix a bug in utils.py where the calculate_tax() function returns wrong results for input > 100000"
    
    for label, tools in [("RAW BASH", RAW_TOOLS), ("PURPOSE-BUILT ACI", ACI_TOOLS)]:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a software engineering agent. Plan how you would approach this task using the available tools. Be specific about which tools you'd call and why."},
                {"role": "user",   "content": f"Task: {task}"}
            ],
            tools=tools,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        print(f"\n{'='*55}")
        rprint(f"[bold]{label}[/bold]")
        print(f"{'='*55}")
        
        if msg.tool_calls:
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                print(f"First action: {tc.function.name}({args})")
        if msg.content:
            print(f"Plan: {msg.content[:200]}")

show_aci_advantage()

rprint(Panel(
    "Key SWE-agent insight: Tool design IS architecture.\n"
    "Raw bash → agent generates wrong/overly broad commands\n"
    "ACI tools → agent uses precise, targeted, safe operations\n\n"
    "Result: 48% improvement in bug-fix success rate",
    title="SWE-agent Lesson", border_style="green"
))